# Zero-Coupon Yield Curve: Bootstrap Walkthrough

This notebook walks through the complete pipeline:
1. Fetch US Treasury CMT par yields from FRED
2. Bootstrap zero rates and discount factors
3. Compare interpolation methods
4. Fit a Nelson-Siegel parametric model
5. Price a fixed-rate bond and compute DV01 / duration
6. Apply parallel shifts and steepener/flattener shocks

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from dotenv import load_dotenv
load_dotenv('../.env')

pd.options.display.float_format = '{:.6f}'.format

## 1. Fetch Treasury par yields from FRED

The FRED DGS series provides **Constant Maturity Treasury (CMT)** yields — synthetic par yields for each tenor, published daily by the US Treasury.

- Bills (1M, 3M, 6M): discount securities → already zero-coupon rates
- Notes/Bonds (1Y–30Y): par yields of hypothetical coupon bonds → must be bootstrapped into zero rates

In [ ]:
from yieldcurve.data.fetcher import fetch_treasury_yields, yields_for_date
from yieldcurve.data.storage import save_yields, load_yields

# Fetch — set FRED_API_KEY in .env or pass api_key= here
df_yields = fetch_treasury_yields(start_date='2000-01-01', end_date='2024-12-31')
save_yields(df_yields)
print(f'Shape: {df_yields.shape}')
df_yields.tail()

In [ ]:
# Pick a curve date
CURVE_DATE = '2024-01-02'

par_yields = yields_for_date(df_yields, CURVE_DATE)  # returns decimal values
print(f'Par yields as of {CURVE_DATE}:')
print((par_yields * 100).round(4).to_string())

## 2. Bootstrap the zero-coupon curve

**Key formulas:**

For T-bills (T ≤ 6M):
$$DF(T) = \frac{1}{1 + c \cdot T}$$

For coupon bonds (T ≥ 1Y), solving iteratively for DF(T):
$$100 = \frac{c}{2} \cdot 100 \sum_{i=1}^{n-1} DF(t_i) + 100 \cdot \left(1 + \frac{c}{2}\right) \cdot DF(T)$$
$$\Rightarrow DF(T) = \frac{100 - \frac{c}{2} \cdot 100 \sum_{i<n} DF(t_i)}{100 \cdot \left(1 + c/2\right)}$$

Annually compounded zero rate:
$$z(T) = DF(T)^{-1/T} - 1$$

In [ ]:
from yieldcurve.curve.bootstrap import BootstrappedCurve

bootstrap = BootstrappedCurve.from_series(par_yields, curve_date=pd.Timestamp(CURVE_DATE))
df_boot = bootstrap.to_dataframe()
df_boot['zero_rate_pct'] = df_boot['zero_rate'] * 100
df_boot['par_yield_pct'] = par_yields.values * 100
df_boot[['tenor', 'par_yield_pct', 'zero_rate_pct', 'discount_factor', 'continuous_rate']].round(6)

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_boot['tenor'], y=df_boot['par_yield_pct'],
                         mode='lines+markers', name='Par yield (CMT)'))
fig.add_trace(go.Scatter(x=df_boot['tenor'], y=df_boot['zero_rate_pct'],
                         mode='lines+markers', name='Bootstrapped zero rate', line=dict(dash='dash')))
fig.update_layout(template='plotly_white', title=f'Par vs Zero Rates — {CURVE_DATE}',
                  xaxis_title='Maturity (years)', yaxis_title='Rate (%)', hovermode='x unified')
fig.show()

In [ ]:
# Discount factors
fig_df = go.Figure()
fig_df.add_trace(go.Scatter(x=df_boot['tenor'], y=df_boot['discount_factor'],
                             mode='lines+markers', name='DF', line=dict(color='steelblue')))
fig_df.update_layout(template='plotly_white', title=f'Discount Factors — {CURVE_DATE}',
                     xaxis_title='Maturity (years)', yaxis_title='Discount Factor')
fig_df.show()

## 3. Interpolation methods

| Method | Description | Forward curve |
|--------|-------------|---------------|
| Linear on zero rates | Simplest; piecewise linear | Discontinuous (step function) |
| Cubic spline on zero rates | Smooth spot curve | Smooth but can oscillate |
| Log-linear on discount factors | Piecewise constant forwards | Flat-forward interpolation |
| Cubic spline on log-DF | Smooth discount + smooth fwd | Best overall for trading use |

In [ ]:
from yieldcurve.curve.interpolation import InterpolatedCurve, InterpolationMethod
from yieldcurve.viz.plots import plot_interpolation_comparison

plot_interpolation_comparison(bootstrap).show()

In [ ]:
# Forward rates show the biggest differences
t_grid = np.linspace(0.25, 30, 200)
step = t_grid[1] - t_grid[0]

fig_fwd = go.Figure()
palette = px.colors.qualitative.Set2
method_labels = {
    InterpolationMethod.LINEAR_ZERO: 'Linear (zero)',
    InterpolationMethod.CUBIC_ZERO: 'Cubic (zero)',
    InterpolationMethod.LOG_LINEAR_DF: 'Log-linear (DF)',
    InterpolationMethod.CUBIC_LOG_DF: 'Cubic log-DF',
}
for i, (method, label) in enumerate(method_labels.items()):
    c = InterpolatedCurve(bootstrap, method)
    fwd = np.array([c.forward_rate(max(t-step, 1e-4), t) for t in t_grid]) * 100
    fig_fwd.add_trace(go.Scatter(x=t_grid, y=fwd, mode='lines', name=label,
                                  line=dict(color=palette[i])))

fig_fwd.update_layout(template='plotly_white', title='Forward Rates by Interpolation Method',
                       xaxis_title='Maturity (years)', yaxis_title='Forward Rate (%)',
                       hovermode='x unified')
fig_fwd.show()

## 4. Nelson-Siegel parametric fit

The Nelson-Siegel model decomposes the yield curve into three factors:

$$r(t) = \beta_0 + (\beta_1 + \beta_2)\frac{\lambda}{t}\left(1 - e^{-t/\lambda}\right) - \beta_2 e^{-t/\lambda}$$

- **β₀**: long-term level (as t→∞)
- **β₁**: short-term component (loads heavily at short end)
- **β₂**: medium-term curvature (hump/trough around t=λ)
- **λ**: decay factor — controls the location of the hump

In [ ]:
from yieldcurve.curve.nelson_siegel import NelsonSiegelCurve, NelsonSiegelSvenssonCurve

ns = NelsonSiegelCurve.fit(bootstrap.tenors, bootstrap.zero_rates)
nss = NelsonSiegelSvenssonCurve.fit(bootstrap.tenors, bootstrap.zero_rates)

print('Nelson-Siegel parameters:')
for k, v in ns.parameter_table().items():
    print(f'  {k}: {v:.6f}')

In [ ]:
t_dense = np.linspace(0.25, 30, 300)
interp = InterpolatedCurve(bootstrap, InterpolationMethod.CUBIC_LOG_DF)

fig_ns = go.Figure()
fig_ns.add_trace(go.Scatter(x=t_dense, y=interp.zero_rate(t_dense)*100,
                             mode='lines', name='Bootstrap (cubic log-DF)'))
fig_ns.add_trace(go.Scatter(x=t_dense, y=ns.zero_rate(t_dense)*100,
                             mode='lines', name='Nelson-Siegel', line=dict(dash='dash')))
fig_ns.add_trace(go.Scatter(x=t_dense, y=nss.zero_rate(t_dense)*100,
                             mode='lines', name='Nelson-Siegel-Svensson', line=dict(dash='dot')))
fig_ns.add_trace(go.Scatter(x=bootstrap.tenors, y=bootstrap.zero_rates*100,
                             mode='markers', name='Bootstrap nodes',
                             marker=dict(symbol='x', size=10, color='black')))
fig_ns.update_layout(template='plotly_white', title='Nelson-Siegel vs Bootstrap',
                      xaxis_title='Maturity (years)', yaxis_title='Zero Rate (%)',
                      hovermode='x unified')
fig_ns.show()

## 5. Bond pricing, DV01, and duration

Using our bootstrapped curve to price a hypothetical 10-year fixed-rate bond.

$$P = \sum_{t} CF_t \cdot DF(t)$$

**Modified Duration:** $MD = \frac{D_{mac}}{1 + y/2}$

**DV01:** $\text{DV01} = MD \times P \times 0.0001$

**Price change approximation:**
$$\Delta P \approx -MD \cdot P \cdot \Delta y + \frac{1}{2} \cdot \text{Convexity} \cdot P \cdot (\Delta y)^2$$

In [ ]:
from yieldcurve.risk.instruments import FixedRateBond

bond = FixedRateBond(coupon_rate=0.04, maturity=10.0)
report = bond.risk_report(interp)

print(f"Price:              ${report['price']:.4f}")
print(f"YTM:                {report['ytm_pct']:.4f}%")
print(f"Macaulay Duration:  {report['macaulay_duration']:.4f} years")
print(f"Modified Duration:  {report['modified_duration']:.4f} years")
print(f"DV01:               ${report['dv01']:.4f} per $100 par")
print(f"Convexity:          {report['convexity']:.4f}")

In [ ]:
# P&L under various rate shocks
shock_sizes = [-200, -100, -50, -25, +25, +50, +100, +200]
rows = []
for bps in shock_sizes:
    est = bond.price_change_estimate(interp, bps)
    rows.append({'Shock (bp)': bps, **{k: round(v, 5) for k, v in est.items()}})

pd.DataFrame(rows)

In [ ]:
# Key-rate DV01 ladder
from yieldcurve.risk.scenarios import ShockEngine
from yieldcurve.viz.plots import plot_dv01_ladder

dv01_ladder = {}
base_price = bond.price(interp)
for lbl in ['1Y', '2Y', '3Y', '5Y', '7Y', '10Y', '20Y', '30Y']:
    shifted = ShockEngine.key_rate_shift(interp, lbl, shift_bps=1.0)
    dv01_ladder[lbl] = bond.price(shifted) - base_price

plot_dv01_ladder(dv01_ladder).show()

## 6. Shock scenarios

Standard rate shock scenarios used in risk management:

| Scenario | 2Y shift | 10Y shift | Interpretation |
|----------|----------|-----------|----------------|
| Parallel +100bp | +100 | +100 | General rate rise |
| Bear steepener | +150 | +50 | Short rates spike (tightening) |
| Bull steepener | -150 | -50 | Short rates fall faster (cuts expected) |
| Bear flattener | +50 | +150 | Long rates spike (term premium) |
| Bull flattener | -50 | -150 | Long rates fall faster (recession fear) |
| Twist | +75 | -75 | Butterfly: 2Y up, 10Y down |

In [ ]:
from yieldcurve.risk.scenarios import ShockEngine, ShockType
from yieldcurve.viz.plots import plot_scenario_curves

shocked_curves = ShockEngine.all_scenarios(interp, magnitude_bps=100)
plot_scenario_curves(interp, shocked_curves).show()

In [ ]:
# Bond P&L by scenario
scenario_pnl = []
for name, sc in shocked_curves.items():
    px_shocked = bond.price(sc)
    scenario_pnl.append({
        'scenario': name.replace('_', ' ').title(),
        'shocked_price': round(px_shocked, 4),
        'pnl': round(px_shocked - base_price, 4),
        'pnl_pct': round((px_shocked - base_price) / base_price * 100, 4),
    })

pd.DataFrame(scenario_pnl)

## 7. Historical yield curve analysis

In [ ]:
from yieldcurve.viz.plots import plot_yield_history, plot_3d_surface

df_hist = load_yields()
plot_yield_history(df_hist, ['2Y', '5Y', '10Y', '30Y']).show()

In [ ]:
# 10Y - 2Y spread (classic recession indicator)
spread = (df_hist['10Y'] - df_hist['2Y']).dropna()
fig_spread = go.Figure()
fig_spread.add_trace(go.Scatter(x=spread.index, y=spread.values, mode='lines',
                                 name='10Y–2Y', fill='tozeroy', line=dict(color='steelblue')))
fig_spread.add_hline(y=0, line_color='red', line_dash='dash', annotation_text='Inversion')
fig_spread.update_layout(template='plotly_white', title='10Y – 2Y Treasury Spread',
                          xaxis_title='Date', yaxis_title='Spread (%)')
fig_spread.show()

In [ ]:
# 3-D yield surface (last 500 trading days)
plot_3d_surface(df_hist.tail(500)).show()